In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path
from datasets import Dataset
from sklearn.metrics import f1_score
import torch

/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.cuda.is_available()

False

In [3]:
torch.backends.mps.is_available()

True

In [4]:
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print (x)
else:
    print ("MPS device not found.")

tensor([1.], device='mps:0')


##### Model Setup

In [5]:
model_id = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [6]:
# tokenizer.model_max_length

#### Data Import

In [6]:
path_to_data_file = Path('./../data/processed/bioasq_embeddings_subset.parquet')
path_to_data_file.exists()

True

In [7]:
df = pd.read_parquet(path_to_data_file)

#### Split into train/test sets

In [8]:
df['combined_text'] = df['question'] + ' ' + df['abstract']
X = df['combined_text']
y = df['label']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

In [10]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train, 
    y_train, 
    test_size=0.1, 
    random_state=42, 
    stratify=y_train
)

#### Tokenize

In [11]:
df_train = pd.DataFrame({
    "text": X_train.tolist(),
    "label": y_train.tolist()
})

df_val = pd.DataFrame({
    "text": X_val.tolist(),
    "label": y_val.tolist()
})

df_test = pd.DataFrame({
    "text": X_test.tolist(),
    "label": y_test.tolist()
})

dataset_train = Dataset.from_pandas(df_train, preserve_index=False)
dataset_val = Dataset.from_pandas(df_val, preserve_index=False)
dataset_test = Dataset.from_pandas(df_test, preserve_index=False)

In [15]:
def tokenize_batch(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding="max_length", # changed from False
        # return_tensors="pt", removed - this causes an error
        # max_length=512
    )

In [16]:
tokenized_train = dataset_train.map(tokenize_batch, batched=True)
tokenized_val = dataset_val.map(tokenize_batch, batched=True)
tokenized_test = dataset_test.map(tokenize_batch, batched=True)

100%|██████████| 1/1 [00:00<00:00, 18.72ba/s]


In [17]:
tokenized_train

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 147
})

In [16]:
# tokenized_train = tokenized_train.rename_column("label", "labels")
# tokenized_val = tokenized_val.rename_column("label", "labels")
# tokenized_test = tokenized_test.rename_column("label", "labels")

In [17]:
# tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
# tokenized_val.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
# tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [18]:
data_collator = DataCollatorWithPadding(tokenizer)

#### Prepare Model Labels

In [19]:
labels = {0, 1}
num_labels = len(labels)

label2id, id2label = dict(), dict()

for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

#### Download and Configure Model

In [20]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=num_labels, 
    label2id=label2id, 
    id2label=id2label
)

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
model.to(mps_device)

ModernBertForSequenceClassification(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      

#### Troubleshooting

In [22]:
# tokenized_train.with_format("python")

In [23]:
# example = tokenized_train[0]
# print(type(example["input_ids"]), len(example["input_ids"]))
# print(type(example["attention_mask"]), len(example["attention_mask"]))

In [22]:
# Metric helper method
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    score = f1_score(
            labels, predictions, labels=labels, pos_label=1, average="weighted"
        )
    return {"f1": float(score) if score == 1 else score}

#### Configure Trainer

In [29]:
training_args = TrainingArguments(
    output_dir="./modernbert-finetuned",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=5,
    learning_rate=5e-5,
    bf16=True, # bfloat16 training 
    optim="adamw_torch_fused", # improved optimizer 
    # logging & evaluation strategies
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

In [30]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    # tokenizer=tokenizer,
    # data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [31]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,No log,0.996529,0.282483
2,0.848800,1.034049,0.752118
3,0.238900,2.988136,0.510181
4,0.238900,0.610173,0.752118
5,0.000100,1.610920,0.680056


TrainOutput(global_step=370, training_loss=0.293996936055152, metrics={'train_runtime': 2847.1684, 'train_samples_per_second': 0.258, 'train_steps_per_second': 0.13, 'total_flos': 4007312269148160.0, 'train_loss': 0.293996936055152, 'epoch': 5.0})

In [32]:
metrics = trainer.evaluate(eval_dataset=tokenized_test)
metrics

{'eval_loss': 0.262993186712265,
 'eval_f1': 0.9047619047619048,
 'eval_runtime': 42.8976,
 'eval_samples_per_second': 0.979,
 'eval_steps_per_second': 0.49,
 'epoch': 5.0}